In [6]:
import sys
import subprocess
import site
import importlib

print("Ansiklopedik QA Scraper ortamı hazırlanıyor...")
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'cloudscraper', 'beautifulsoup4', 'pandas', 'tqdm'])
    
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
        
    importlib.invalidate_caches()
    
    import cloudscraper
    from bs4 import BeautifulSoup
    import pandas as pd
    from tqdm.notebook import tqdm
    
    print("✅ TEST BAŞARILI: Veri madenciliği modülleri hazır!")
except Exception as e:
    print(f"\n❌ HATA OLUŞTU: {e}")

Ansiklopedik QA Scraper ortamı hazırlanıyor...
✅ TEST BAŞARILI: Veri madenciliği modülleri hazır!


In [7]:
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from tqdm.notebook import tqdm
from urllib.parse import urljoin

OUTPUT_FILE = "data/06_hospital_qa_dataset.csv"

if not os.path.isfile(OUTPUT_FILE):
    pd.DataFrame(columns=["source", "question", "answer", "url"]).to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

scraper = cloudscraper.create_scraper()

# MedicalPark Sağlık Rehberi (Hastalıklar A'dan Z'ye)
BASE_URL = "https://www.medicalpark.com.tr/saglik-rehberi"
DOMAIN = "https://www.medicalpark.com.tr"

article_links = set()
dataset = []

print("1. AŞAMA: Hastane Ansiklopedisi (Sağlık Rehberi) makale havuzu taranıyor...")

try:
    # İlk 15 makale listeleme sayfasını geziyoruz (Sitedeki içerik sayısına göre artırılabilir)
    for page in range(1, 16):
        url = f"{BASE_URL}?page={page}" if page > 1 else BASE_URL
        
        try:
            resp = scraper.get(url, timeout=15)
            if resp.status_code == 200:
                soup = BeautifulSoup(resp.text, 'html.parser')
                
                for a in soup.find_all('a', href=True):
                    link = a['href']
                    # Sadece sağlık rehberi makalelerini topla
                    if '/saglik-rehberi/' in link and link.count('/') >= 2:
                        full_link = link if link.startswith('http') else urljoin(DOMAIN, link)
                        article_links.add(full_link)
            time.sleep(1) # Ban koruması
        except:
            continue
            
    print(f"✅ Toplam {len(article_links)} adet klinik makale bulundu!")
    print("2. AŞAMA: Makaleler Yapay Zeka için Soru-Cevap (QA) Formatına Dönüştürülüyor...")

    with tqdm(total=len(article_links), desc="Ansiklopedi Okunuyor") as pbar:
        for url in list(article_links):
            try:
                resp = scraper.get(url, timeout=15)
                if resp.status_code == 200:
                    soup = BeautifulSoup(resp.text, 'html.parser')
                    
                    # Makalenin içindeki alt başlıkları (H2 ve H3) soru olarak hedef alıyoruz
                    headers = soup.find_all(['h2', 'h3'])
                    
                    for header in headers:
                        question = header.text.strip()
                        
                        # Mantıklı bir başlık uzunluğu filtrelemesi
                        if 5 < len(question) < 120:
                            
                            # Başlığın altındaki paragrafları (Cevabı) topla (Bir sonraki başlığa kadar)
                            answer_paragraphs = []
                            nextNode = header
                            while True:
                                nextNode = nextNode.find_next_sibling()
                                # Eğer sayfa bittiyse veya yeni bir başlık geldiyse okumayı durdur
                                if nextNode is None or nextNode.name in ['h2', 'h3', 'h1']:
                                    break
                                # Eğer paragraf ise metni al
                                if nextNode.name == 'p':
                                    answer_paragraphs.append(nextNode.text.strip())
                                    
                            answer = " ".join(answer_paragraphs).strip()
                            
                            # Eğer cevap doyurucuysa (150 karakterden büyükse) veritabanına ekle
                            if len(answer) > 150:
                                # Modelin soruyu anlaması için başlığı soru formatına zorla
                                if not question.endswith('?'):
                                    if "nedir" not in question.lower() and "neler" not in question.lower():
                                        question = f"{question} hakkında detaylı bilgi verir misiniz?"
                                
                                dataset.append({
                                    'source': 'MedicalPark_Encyclopedia',
                                    'question': question,
                                    'answer': answer,
                                    'url': url
                                })
                                
                    # RAM şişmesini önlemek için 50 soruda bir kaydet
                    if len(dataset) >= 50:
                        pd.DataFrame(dataset).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")
                        dataset = []
                        
                time.sleep(1.5) 
                
            except Exception as e:
                pass
                
            pbar.update(1)

    # Kalan verileri kaydet
    if dataset:
        pd.DataFrame(dataset).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")
        
    print(f"\n🎉 İşlem Tamam! Profesör onaylı Soru-Cevap çiftleri '{OUTPUT_FILE}' dosyasına başarıyla kaydedildi.")

except Exception as e:
    print(f"Kritik Hata: {e}")

1. AŞAMA: Hastane Ansiklopedisi (Sağlık Rehberi) makale havuzu taranıyor...
✅ Toplam 177 adet klinik makale bulundu!
2. AŞAMA: Makaleler Yapay Zeka için Soru-Cevap (QA) Formatına Dönüştürülüyor...


Ansiklopedi Okunuyor:   0%|          | 0/177 [00:00<?, ?it/s]


🎉 İşlem Tamam! Profesör onaylı Soru-Cevap çiftleri 'data/06_hospital_qa_dataset.csv' dosyasına başarıyla kaydedildi.
